# Topical Lectures, Controls 
## 2D simulator, part 1: keyboard flying is hard
Andreas Freise, Bas Swinkels 30.10.2026

Before we fly the real Crazyflie we learn feedback control on a simulated drone. The simulator is a model of a drone in a vertical plane: it can move sideways and up and down, and it can tilt, driven by two rotors, left and right. We control it by setting the throttle of the two rotors, and the model computes the motion of the drone in steps of 1/60 s. The model code is stored in the file `module.py`. We use it as a black box to experiment with, but you are welcome to read the code.

The simulator notebooks build up the control step by step: here we fly the drone by hand, then we measure its properties, build feedback loops for tilt, height and position, add a velocity loop and race the result through a track. The same loop structure flies the real drone in the afternoon.

This first notebook has two aims: a) to get familiar with the notebook format and b) to learn that flying a drone without any control loop is difficult. 

In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

# This is a local file containing the drone code. You are supposed to manage the drone
# as a 'black box', but feel free to study the code if you want.
import module

# For the interactive plots we need the QT backend
%matplotlib qt

The ```module``` package contains code which models a drone and separately a pre-defined plot that can be used for interactive plots.

The main interaction with the drone happens through the following steps:
 * create a drone with: ```drone = module.Drone(...)```
 * set the voltage of the left and right rotor motors with ```self.drone.set_V(V_left, V_right)```
 * model one time step (1/60 sec) with ```results = self.drone.update()```. ```results``` will contain the current data from the drone such as position and velocities, see below.

Typically for each task you create a new drone model and then run a loop in which you continuously set the rotor voltages and then run one time step forward. You can either plot the data received in ```results``` in an interactive way, or store it in an array for a later analysis.

In [ ]:
# An example code that applies manual signals to the drone:
# the voltages sent to the left and right rotor can be changed with the keys

class Control():
    def __init__(self, _drone):
        self.drone = _drone
        self.plotter = module.Plotter(self.drone, target_fps=60)

        # step values for keyboard control. The drone parameters give
        # roughly 1160 rad/s^2 of angular acceleration per unit of
        # `offset`, so key_dF0 needs to be small.
        self.key_dF = 0.01      # throttle units, +/- per w/s press
        self.key_dF0 = 5e-5     # throttle units, +/- per a/d press

        self.stop = False
        self.reset()

    def reset(self):
        # Start at hover so the drone is not in free-fall before the
        # first keypress. V_hover = 0.5297 throttle per motor for the
        # default Crazyflie-scale drone (M*g / 2 / amp).
        self.F = 0.5297
        self.offset = 0
        self.set_F2leftright()
        self.plotter.reset()

    def set_F2leftright(self):
        # computing the left and right voltages from F and offset
        F_right = self.F - self.offset
        F_left =  self.F + self.offset
        self.V_left  = F_left
        self.V_right = F_right

    def physics_step(self):
        """One simulation tick. Runs at drone.deltaT (60 Hz) of wall-clock time."""
        self.drone.set_V(self.V_left, self.V_right)
        self.drone.update()

    def render_step(self):
        """One screen redraw. Runs at the plotter's target_fps."""
        self.plotter.update_display()

    def key_control_simple(self):
        # initialising interactive plot
        self.stop = False
        self.drone.stop = False
        # connecting key event manager
        self.plotter.fig.canvas.mpl_disconnect(self.plotter.fig.canvas.manager.key_press_handler_id)
        self.cid = self.plotter.fig.canvas.mpl_connect('key_press_event', self.on_key_simple)

        # Start animation (blocks until stopped)
        self.plotter.start_animation(
            self.physics_step,
            self.render_step,
            lambda: self.stop
        )

        # disconnecting key event manager
        self.plotter.fig.canvas.mpl_disconnect(self.cid)

    def on_key_simple(self, event):
        if event.key == 'escape':
            print('Stopping!')
            self.stop = True
        elif event.key == 'w':
            self.F += self.key_dF
        elif event.key == 's':
            self.F -= self.key_dF
        elif event.key == 'a':
            self.offset -= self.key_dF0
        elif event.key == 'd':
            self.offset += self.key_dF0
        self.set_F2leftright()


In [ ]:
# Create the drone object in test mode 
# (Only use the test mode in this notebook!)
drone = module.Drone(test=True)

# set a target for us to reach at y=1.0 m, z=3.0 m with a radius of 0.1 m
drone.set_targets(np.array([[1.5, 2, 0.1]]))

# initialising our control object
control = Control(drone)


Now start the keyboard control in the next cell and try to reach the target (red circle) by flying the drone manually.

In [ ]:
# This starts the interactive plot, after running this cell, click on the plot and 
# use the WASD keys to fly the drone. Use ESC to stop.
control.key_control_simple()

In [ ]:
control.plotter.close()

In [ ]:
drone.reset()
control.reset()